# 01 - intro

In [7]:
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the program, can I still enroll"

# Although they are exactly the same thing semantically, they have very different words
# Because examples like this, we need to have something else besides keyword search

# 02 - embeddings

In [8]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
v1 = model.encode(q1)
v2 = model.encode(q2)
v2.dot(v1)

np.float32(0.62177926)

In [10]:
q3 = "How to install Docker on Windows?"
v3 = model.encode(q3)
v3.dot(v1)

np.float32(-0.1180307)

# 03 - embeddings dataset

In [11]:
from pathlib import Path
import sys

parent = Path.cwd().parent
sys.path.append(str(parent))

from lesson1_agentic_rag.ingest import load_faq_data

In [12]:
documents = load_faq_data()

In [13]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [14]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [15]:
from tqdm.auto import tqdm 

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [16]:
vectors[10].shape

(384,)

# 04 - vector search

In [17]:
v1.dot(vectors[10])

np.float32(0.31346333)

In [18]:
scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

In [19]:
import numpy as np
X = np.array(vectors)

# One way of getting the most similar result with my query (v1), is doing vector
# multiplication in a for loop and then getting the answer-question with the highest value
# instead of it, we will do matrix multiplication

In [20]:
X

array([[-0.02670622, -0.12245756,  0.01594418, ..., -0.00230645,
        -0.11218391, -0.02365551],
       [-0.01099553, -0.11074745, -0.02536936, ...,  0.09022227,
        -0.02697366,  0.01975676],
       [-0.08896549, -0.06128184,  0.00775605, ...,  0.0405971 ,
         0.0047928 , -0.0274594 ],
       ...,
       [-0.03652925,  0.01415433, -0.06838643, ...,  0.04316785,
         0.08105534, -0.02148626],
       [-0.13091588, -0.06990597, -0.00931879, ..., -0.00044345,
        -0.01285729,  0.01426925],
       [-0.07984771,  0.01926989,  0.02544983, ..., -0.03368025,
        -0.01884021,  0.05837058]], shape=(1350, 384), dtype=float32)

In [21]:
scores = X.dot(v1) # same but much faster than the previous loop

In [22]:
scores

array([ 0.4130352 ,  0.26119855,  0.6088087 , ..., -0.04739505,
        0.08148281, -0.0235371 ], shape=(1350,), dtype=float32)

In [23]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(538), np.float32(0.831779))

In [24]:
documents[538]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [25]:
idx_ranked = np.argsort(scores)[-5:][::-1] # [::-1] reverses the documents, so [1, 2, 3] becomes [3, 2, 1]
idx_ranked, scores[idx_ranked]

(array([538, 907, 625,   2, 503]),
 array([0.831779  , 0.6845695 , 0.61755615, 0.6088087 , 0.58479655],
       dtype=float32))

In [26]:
for idx in idx_ranked:
    print(scores[idx])
    print(documents[idx]['question'])
    print()

0.831779
I just discovered the course. Can I still join?

0.6845695
The course has already started. Can I still join it?

0.61755615
Course - Can I still join the course after the start date?

0.6088087
Course: Can I still join the course after the start date?

0.58479655
Am I too late to join the course?



# 05 - minsearch vector

In [27]:
from minsearch import VectorSearch

In [28]:
vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [30]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [31]:
results = vindex.search(
    v1,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

# 06 - rag vector

In [34]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-27 16:40:15--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-27 16:40:16 (33.9 MB/s) - ‘rag_helper.py’ saved [2134/2134]



In [35]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [37]:
from pathlib import Path
import sys

parent = Path.cwd().parent
sys.path.append(str(parent))

from lesson1_agentic_rag.ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [38]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [39]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while submissions are still open.'

In [40]:

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [41]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [42]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. You can start learning and submitting homework while the form is open. If you want a certificate, make sure to submit your project before submissions close.'

# 06 - sqlite search

In [ ]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf", # how we use NN - ivf: 10K-500K vectors, K-means clustering
    db_path="faq_vectors2.db"
)

In [45]:
vs_index.fit(vectors, documents)

In [50]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5, filter_dict={'course':'llm-zoomcamp'})

In [51]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

In [52]:
vs_index.close()